# UMAP by Condition, Over Time (Cell + Nucleus)

For each overlapping timepoint between `naive` and `A1_1uM_vem`, shows the
shared UMAP embedding (fit directly on CAJAL's Gromov-Wasserstein
distances - no Isomap step) with a kernel density overlay per condition.
Every subclass (dying, persisting, slow/fast proliferator) is included -
this is the broadest view before narrowing to proliferators only
(`02_proliferators_vem_vs_naive.ipynb`) or to vem's own subclasses
(`03_vem_subclasses_over_time.ipynb`).

Requires `main.py` to have been run first (produces `selected_metadata.csv`,
`cell_umap.npz`, `nuc_umap.npz` under `output/cajal_output/`).

In [ ]:
import os
import sys

import matplotlib.pyplot as plt

# assumes Jupyter's cwd is this notebook's own directory (analysis/), so the
# repo root - and config.py - is one level up
REPO_ROOT = os.path.abspath('..')
sys.path.insert(0, REPO_ROOT)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR are relative to the repo root, not
# this notebook's own cwd - re-anchor them to an absolute path once, here
config.OUTPUT_DIR       = os.path.join(REPO_ROOT, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(REPO_ROOT, config.CAJAL_OUTPUT_DIR)

from helpers.load_data import load_full_dataset
from helpers.density_plots import export_kde_by_frame_pdf

PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

## Plot: UMAP with per-condition kernel density, one PDF page per frame

In [ ]:
CONDITION_COLORS = {'naive': 'tab:blue', 'A1_1uM_vem': 'tab:red'}

def process_modality(modality):
    '''
    Load this modality's embedding + metadata (every subclass, every
    condition) and export one PDF page per overlapping frame, with a
    KDE contour per condition.

    Inputs:
        modality: 'cell' or 'nuc'
    Outputs:
        df: the full merged dataset for this modality (in-memory only)
    '''
    df = load_full_dataset(modality)
    print(f"[{modality}] {len(df)} cells across {df['frame'].nunique()} frames, "
          f"{df['condition'].nunique()} conditions")

    pdf_path = os.path.join(PLOTS_DIR, f'umap_by_condition_{modality}.pdf')
    export_kde_by_frame_pdf(df, 'condition', CONDITION_COLORS, pdf_path, f'{modality} - all subclasses')

    return df

## Run: cell morphology

In [ ]:
cell_df = process_modality('cell')
cell_df.head()

## Run: nuclear morphology

In [ ]:
nuc_df = process_modality('nuc')
nuc_df.head()